# Reading and writing Iceberg geometry tables with Sedona-Sail

Wherobots catalogs are Apache Iceberg catalogs, and since Iceberg format version 3 a table column can be a real **`geometry`** type rather than a blob of WKB. WherobotsDB, SedonaDB and sedona-sail all read that type natively, so a table written by one is a typed geometry table for the others.

This notebook, on the sedona-sail runtime:

1. connects a Sail session to the workspace's catalogs;
2. creates a namespace and a table with a `geometry` column in the `org_catalog` catalog;
3. writes rows into it from SQL and from a DataFrame;
4. reads them back with spatial SQL;
5. drops the table again.

Everything below runs in this notebook's process: Sail is a Rust-native Spark Connect server with the SedonaDB spatial kernels, and no Spark executors are started.

## Connect Sail to your catalogs

WherobotsDB finds the workspace catalogs because the runtime writes every one of them into `$SPARK_CONF_DIR/spark-defaults.conf` as `spark.sql.catalog.<name>.*`, together with a refresh token. Sail reads a different configuration format, so the helper below does the same wiring by hand: it reads those Spark settings, mints a short-lived catalog token the way WherobotsDB does, hands each catalog to Sail as an Iceberg REST catalog (`SAIL_CATALOG__LIST`), and keeps the token fresh in the background. The session object is named `sedona`, as in the other examples.

In [ ]:
import os
import re
import sys
import json
import time
import threading
import urllib.parse
import urllib.request

TOKEN_DIR = "/tmp/wbc-catalog-tokens"


def read_spark_defaults():
    path = os.path.join(os.environ.get("SPARK_CONF_DIR", "/opt/spark/conf"), "spark-defaults.conf")
    conf = {}
    for line in open(path):
        m = re.match(r"^\s*([^#\s=]+)[\s=]+(.*?)\s*$", line)
        if m:
            conf.setdefault(m.group(1), m.group(2))
    return conf


def wherobots_catalogs(conf):
    """Every catalog served by the Wherobots catalog service, keyed by name."""
    out = {}
    for key, uri in conf.items():
        m = re.match(r"^spark\.sql\.catalog\.([^.]+)\.uri$", key)
        prefix = f"spark.sql.catalog.{m.group(1)}." if m else None
        if m and conf.get(prefix + "type") == "rest" and "wherobots.com/" in uri:
            out[m.group(1)] = {
                "uri": uri.rstrip("/"),
                "token_uri": conf.get(prefix + "oauth2-server-uri", uri.rstrip("/") + "/v1/oauth/tokens"),
                "refresh_token": conf.get(prefix + "header.Ste-Refresh-Token"),
            }
    return out


def mint_token(catalog):
    """A one-hour bearer token for one catalog, minted exactly as WherobotsDB does."""
    form = urllib.parse.urlencode({"grant_type": "client_credentials", "scope": "catalog", "client_secret": "true"}).encode()
    headers = {"Content-Type": "application/x-www-form-urlencoded", "Ste-Refresh-Token": catalog["refresh_token"]}
    with urllib.request.urlopen(urllib.request.Request(catalog["token_uri"], data=form, headers=headers), timeout=30) as r:
        return json.loads(r.read())["access_token"]


def write_token(path, token):
    with open(path + ".tmp", "w") as f:
        f.write(token)
    os.chmod(path + ".tmp", 0o600)
    os.replace(path + ".tmp", path)


catalogs = wherobots_catalogs(read_spark_defaults())
os.makedirs(TOKEN_DIR, mode=0o700, exist_ok=True)
entries = ['{type="memory", name="spark_catalog", initial_database=["default"]}']
for name, catalog in sorted(catalogs.items()):
    token_file = os.path.join(TOKEN_DIR, name)
    write_token(token_file, mint_token(catalog))
    entries.append(f'{{type="iceberg-rest", name="{name}", uri="{catalog["uri"]}", bearer_access_token_file="{token_file}"}}')

# Sail reads its configuration when the server starts, so set it first.
os.environ["SAIL_CATALOG__LIST"] = "[" + ", ".join(entries) + "]"
os.environ["SAIL_CATALOG__DEFAULT_CATALOG"] = "org_catalog"
os.environ.setdefault("AWS_REGION", "us-west-2")


def refresh_tokens():
    # Sail re-reads the token file on every request, so rewriting it is enough.
    while True:
        time.sleep(45 * 60)
        for name, catalog in catalogs.items():
            write_token(os.path.join(TOKEN_DIR, name), mint_token(catalog))


threading.Thread(target=refresh_tokens, daemon=True).start()

import pysail
from pysail.spark import SparkConnectServer
from pyspark.sql.connect.session import SparkSession

server = SparkConnectServer(ip="127.0.0.1", port=0)
server.start(background=True)
host, port = server.listening_address
sedona = SparkSession.builder.remote(f"sc://{host}:{port}").create()
print(f"pysail {pysail.__version__} · {len(catalogs)} catalogs wired: {', '.join(sorted(catalogs))}")

## Browse the catalog

The workspace's catalogs are now ordinary Spark catalogs: `org_catalog` is your organization's, and the Wherobots open and pro data catalogs are there too.

In [ ]:
print([c.name for c in sedona.catalog.listCatalogs()])
sedona.sql("SHOW DATABASES IN org_catalog").show(10, truncate=False)

## Create a namespace and a geometry table

The table gets an Iceberg V3 `geometry(OGC:CRS84)` column — longitude/latitude, the same CRS as the `geometry` columns WherobotsDB writes. The namespace is created with plain SQL. The table itself is registered with one call to the catalog's Iceberg REST API: sedona-sail's own `CREATE TABLE` still records geometry columns as `binary` in the catalog, and that is the one thing this notebook needs that Sail cannot yet do by itself. Everything after this cell is Sail.

Two practicalities, both worth knowing when you work with the catalog from Sail:

- `TABLE_LOCATION_ROOT` is where the table's files live — a folder under your organization's managed storage (open **Data → File Browser** in Wherobots Studio, or copy the location of any table already in the catalog).
- The catalog **does not delete a table's files when the table is dropped** (`DROP TABLE … PURGE` is accepted but has no effect), and a table re-created at a location that still holds an older table's metadata fails on its first write. So the notebook drops *and* purges — `drop_table_and_purge` below — which also makes it safe to run again.

In [ ]:
import boto3

NAMESPACE = "sail_examples"
TABLE = "sf_landmarks"
FULL_NAME = f"org_catalog.{NAMESPACE}.{TABLE}"
TABLE_LOCATION_ROOT = "s3://wbts-wbc-rcv7vl73oy/djrm9bs9uf/data/catalogs/org_catalog/sail_examples"
TABLE_LOCATION = f"{TABLE_LOCATION_ROOT}/{TABLE}"


def drop_table_and_purge(full_name, location):
    """Drop the catalog entry, then delete the files the catalog leaves behind."""
    sedona.sql(f"DROP TABLE IF EXISTS {full_name}")
    bucket, prefix = location[len("s3://"):].split("/", 1)
    s3 = boto3.client("s3")
    objects = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/").get("Contents", [])
    if objects:
        s3.delete_objects(Bucket=bucket, Delete={"Objects": [{"Key": o["Key"]} for o in objects]})
    return len(objects)


sedona.sql(f"CREATE NAMESPACE IF NOT EXISTS org_catalog.{NAMESPACE}")
purged = drop_table_and_purge(FULL_NAME, TABLE_LOCATION)
print(f"cleared {purged} leftover file(s) under {TABLE_LOCATION}")

org_catalog = catalogs["org_catalog"]
request = {
    "name": TABLE,
    "location": TABLE_LOCATION,
    "schema": {"type": "struct", "schema-id": 0, "fields": [
        {"id": 1, "name": "id", "required": False, "type": "long"},
        {"id": 2, "name": "name", "required": False, "type": "string"},
        {"id": 3, "name": "geom", "required": False, "type": "geometry(OGC:CRS84)"},
    ]},
    "properties": {"format-version": "3"},   # geometry columns need Iceberg V3; the catalog does not infer it
}
headers = {"Authorization": f"Bearer {open(os.path.join(TOKEN_DIR, 'org_catalog')).read()}", "Content-Type": "application/json"}
with urllib.request.urlopen(urllib.request.Request(f"{org_catalog['uri']}/v1/namespaces/{NAMESPACE}/tables",
                                                   data=json.dumps(request).encode(), headers=headers), timeout=30) as r:
    metadata = json.loads(r.read())["metadata"]
print(f"created {FULL_NAME}: Iceberg format version {metadata['format-version']}, "
      f"columns {[(f['name'], f['type']) for f in metadata['schemas'][-1]['fields']]}")
sedona.table(FULL_NAME).printSchema()

## Write rows

Rows go in with ordinary `INSERT` statements — the geometry values come from `ST_GeomFromText` with `ST_SetSRID(…, 4326)`, so they carry the table's CRS — or from a DataFrame appended by name. A row with no geometry is inserted through `INSERT … SELECT`, which lets the `NULL` take the column's type.

In [ ]:
sedona.sql(f"""
INSERT INTO {FULL_NAME} VALUES
  (1, 'Ferry Building',   ST_SetSRID(ST_GeomFromText('POINT(-122.3937 37.7955)'), 4326)),
  (2, 'Golden Gate Park', ST_SetSRID(ST_GeomFromText('POLYGON((-122.5108 37.7649, -122.4544 37.7649, -122.4544 37.7749, -122.5108 37.7749, -122.5108 37.7649))'), 4326))
""")
sedona.sql(f"INSERT INTO {FULL_NAME} SELECT 3, 'somewhere in the fog', NULL")

market_street = sedona.sql("""
SELECT 4 AS id, 'Market Street' AS name,
       ST_SetSRID(ST_GeomFromText('LINESTRING(-122.3937 37.7946, -122.4194 37.7749)'), 4326) AS geom
""")
market_street.writeTo(FULL_NAME).append()

print(f"{sedona.table(FULL_NAME).count()} rows in {FULL_NAME}")

## Read it back with spatial SQL

The column comes back as a geometry, so every `ST_*` function applies to it directly — no `ST_GeomFromWKB` needed. `printSchema()` shows what Sail sees; the catalog shows the same schema to WherobotsDB.

One thing to know: SedonaDB is strict about coordinate reference systems. The column carries CRS84, so a literal geometry used against it must carry the same CRS — hence the `ST_SetSRID(…, 4326)` around the filter polygon; comparing a tagged geometry with an untagged one is an error, not a silent guess.

In [ ]:
sedona.sql(f"""
SELECT id, name, ST_GeometryType(geom) AS type, ST_AsText(geom) AS wkt
FROM {FULL_NAME} ORDER BY id
""").show(truncate=False)

downtown = "POLYGON((-122.43 37.76, -122.38 37.76, -122.38 37.80, -122.43 37.80, -122.43 37.76))"
sedona.sql(f"""
SELECT name, ROUND(ST_Length(geom) * 111, 2) AS approx_km
FROM {FULL_NAME}
WHERE ST_Intersects(geom, ST_SetSRID(ST_GeomFromText('{downtown}'), 4326))
ORDER BY name
""").show(truncate=False)

sedona.table(FULL_NAME).printSchema()

## The same table from WherobotsDB

Nothing about the table is Sail-specific: it is a format-version-3 Iceberg table registered in your catalog with a `geometry(OGC:CRS84)` column. From a WherobotsDB notebook, `sedona.table("org_catalog.sail_examples.sf_landmarks")` reads `geom` as a geometry, and `ST_AsText(geom)` returns the same four rows.

## Clean up

Drop the table — and purge its files, since the catalog leaves them behind — then stop the session. The namespace stays; `CREATE NAMESPACE IF NOT EXISTS` above is idempotent, so the notebook can be run again.

In [ ]:
drop_table_and_purge(FULL_NAME, TABLE_LOCATION)
sedona.stop()
server.stop()